# 모터 원천데이터 PNG 연결 분석

연속된 PNG 시퀀스를 이어 붙였을 때 (차량 × 고장유형 × 센서)별로 CSV가 몇 개 생성되는지 사전 집계.

**파일명 형식**: `22_07_28_12_15_20_01_zsplit005_001_Current_U.png`
- `22_07_28_12_15_20_01_zsplit005` → group_key (같은 측정 세션)
- `001` → 시퀀스 번호
- `Current_U` → 센서

**연결 규칙**: 동일 group_key에서 시퀀스 번호가 연속(1→2→3…)이면 하나의 CSV로 합쳐짐.

In [ ]:
import re
from pathlib import Path
from collections import defaultdict

import pandas as pd

In [ ]:
# ── 경로 설정 ──────────────────────────────────────────────────────────────────
# 전체 데이터 루트 (추후 채울 것)
FULL_DATA_ROOT = Path(r"")

# 샘플 데이터 루트 (검증용)
SAMPLE_DATA_ROOT = Path(r"샘플데이터/Sample/01.원천데이터/1.모터_감속기")

# 실제 실행할 경로 선택
DATA_ROOT = SAMPLE_DATA_ROOT if not FULL_DATA_ROOT.as_posix() else FULL_DATA_ROOT
print(f"사용 경로: {DATA_ROOT}")

In [ ]:
# ── 파일명 파싱 ────────────────────────────────────────────────────────────────
# 형식: {group_key}_{seq:03d}_{sensor}.png
# group_key = {timestamp}_{vehicle_id}_zsplit{NNN}
FNAME_PATTERN = re.compile(
    r"^(.+_zsplit\d+)_(\d{3})_(.+)\.png$"
)

def parse_raw_filename(fname: str):
    m = FNAME_PATTERN.match(fname)
    if not m:
        return None
    group_key, seq_str, sensor = m.groups()
    return group_key, int(seq_str), sensor

In [ ]:
# ── 전체 PNG 수집 ──────────────────────────────────────────────────────────────
# 디렉터리 구조: DATA_ROOT / {vehicle} / {fault_type} / {date} / {sensor} / *.png

records = []  # (vehicle, fault_type, sensor, group_key, seq)
parse_errors = []

for png_path in DATA_ROOT.rglob("*.png"):
    parsed = parse_raw_filename(png_path.name)
    if parsed is None:
        parse_errors.append(str(png_path))
        continue

    group_key, seq, sensor_fname = parsed

    # 경로에서 vehicle / fault_type 추출
    rel_parts = png_path.relative_to(DATA_ROOT).parts
    # 예: ('IONIQ', 'DEMAG', '0728', 'Current_U', '22_07_...png')
    if len(rel_parts) < 5:
        parse_errors.append(str(png_path))
        continue

    vehicle, fault_type, _date, sensor_dir = rel_parts[:4]

    records.append({
        "vehicle":    vehicle,
        "fault_type": fault_type,
        "sensor":     sensor_dir,
        "group_key":  group_key,
        "seq":        seq,
    })

print(f"파싱 성공: {len(records):,}개 / 실패: {len(parse_errors)}개")
if parse_errors:
    print("  실패 예시:", parse_errors[:3])

In [ ]:
# ── 연속 시퀀스 그룹 → CSV 개수 계산 ──────────────────────────────────────────

def count_consecutive_runs(seq_list: list[int]) -> int:
    """정렬된 정수 리스트에서 연속 구간 수를 반환."""
    if not seq_list:
        return 0
    nums = sorted(set(seq_list))
    runs = 1
    for a, b in zip(nums, nums[1:]):
        if b != a + 1:
            runs += 1
    return runs


# (vehicle, fault_type, sensor, group_key) → [seq, ...]
group_seqs = defaultdict(list)
for r in records:
    key = (r["vehicle"], r["fault_type"], r["sensor"], r["group_key"])
    group_seqs[key].append(r["seq"])

# (vehicle, fault_type, sensor) → CSV 개수 합산
csv_counts = defaultdict(int)
for (vehicle, fault_type, sensor, _gk), seqs in group_seqs.items():
    csv_counts[(vehicle, fault_type, sensor)] += count_consecutive_runs(seqs)

print(f"총 group_key 수: {len(group_seqs):,}")
print(f"총 예상 CSV 수: {sum(csv_counts.values()):,}")

In [ ]:
# ── 결과 테이블 ────────────────────────────────────────────────────────────────
rows = [
    {"vehicle": v, "fault_type": ft, "sensor": s, "csv_count": cnt}
    for (v, ft, s), cnt in sorted(csv_counts.items())
]
df = pd.DataFrame(rows)

print("\n[차량 × 고장유형 × 센서별 예상 CSV 개수]")
print(df.to_string(index=False))

print("\n[고장유형 × 센서별 소계]")
subtotal = df.groupby(["fault_type", "sensor"])["csv_count"].sum().reset_index()
print(subtotal.to_string(index=False))

print("\n[차량별 소계]")
by_vehicle = df.groupby("vehicle")["csv_count"].sum().reset_index()
print(by_vehicle.to_string(index=False))

print(f"\n합계: {df['csv_count'].sum():,}개")

In [ ]:
# ── (선택) group_key 단위 상세 확인 ───────────────────────────────────────────
detail_rows = []
for (vehicle, fault_type, sensor, gk), seqs in sorted(group_seqs.items()):
    nums = sorted(set(seqs))
    runs = count_consecutive_runs(nums)
    detail_rows.append({
        "vehicle":    vehicle,
        "fault_type": fault_type,
        "sensor":     sensor,
        "group_key":  gk,
        "seq_list":   nums,
        "png_count":  len(seqs),
        "csv_runs":   runs,
    })

df_detail = pd.DataFrame(detail_rows)

# 비연속 구간이 있는 케이스만 표시 (연속이 끊긴 곳 확인)
non_consecutive = df_detail[df_detail["csv_runs"] > 1]
print(f"연속이 끊긴 group_key: {len(non_consecutive)}개")
if not non_consecutive.empty:
    display(non_consecutive[["vehicle", "fault_type", "sensor", "group_key", "seq_list", "csv_runs"]])

df_detail.head(10)